# Experiment Diagnostics and Regression

This notebook demonstrates diagnostics for an experiment assignment and an adjusted binary-outcome model using the existing ShopFlow event table. It opens the SQLite database in read-only mode. Results are observational diagnostics on synthetic data; they do not establish real-world causal effects.

## Interpretation guardrails
- Count unique assigned users, not event rows, for sample-ratio checks.
- Check each user's group is consistent and each user belongs to one group.
- Pre-treatment covariates can be used for balance diagnostics and precision adjustment; do not adjust for post-treatment variables.
- Regression does not fix non-random assignment, missing outcomes, sample-ratio mismatch, peeking, or unmeasured confounding.
- If the dataset is simulated, any coefficient or p-value is a teaching example, not evidence about actual customers.

In [ ]:
import sys
from pathlib import Path

# Make repository imports work when Jupyter starts from the notebooks/ directory.
search_from = Path.cwd().resolve()
repo_root = next(
    (
        candidate
        for candidate in (search_from, *search_from.parents)
        if (candidate / "src" / "experimentation.py").is_file()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate repository root containing src/experimentation.py"
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import sqlite3

import pandas as pd

from src.experimentation import (
    adjusted_logistic_regression,
    holm_bonferroni,
    sample_ratio_mismatch,
)

## Exposure and outcome validation limitations

The current synthetic event table records experiment-group labels on event
rows, but it does not establish a dedicated, reliable exposure event for the
proposed product-page treatment. Assignment must not be treated as proof of
exposure.

The existing adjusted regression uses a user-level indicator for any purchase
event. It does not validate the proposed session-level product-view-to-add-to-cart
metric. Before evaluating a real experiment, define and validate the exposure
event, eligibility rules, attribution window, and outcome logic. Report assigned
users and exposed users separately when reliable exposure data is available.

These checks demonstrate diagnostics on synthetic data; they do not establish
that a real experiment was correctly randomized or measured.

## 1. Read events without modifying the database

In [ ]:
db_path = (repo_root / 'data' / 'product_analytics.db').resolve()
if not db_path.is_file():
    raise FileNotFoundError(f'Database not found at expected repository path: {db_path}')
with sqlite3.connect(f'{db_path.as_uri()}?mode=ro', uri=True) as connection:
    events = pd.read_sql_query(
        'SELECT user_id, timestamp, event_name, device, traffic_source, experiment_group FROM events',
        connection,
    )
events.head()


## 2. Validate assignment consistency and sample ratio
This checks unique users and whether each user has one consistent group. If a user appears in multiple groups, stop and investigate the assignment data before interpreting treatment effects.

In [ ]:
# Validate assignment consistency and completeness at user level.
valid_groups = {"control", "treatment"}

user_group_counts = (
    events.groupby("user_id")["experiment_group"]
    .nunique(dropna=True)
)

multi_group_users = int((user_group_counts > 1).sum())

missing_group_users = int(
    events.loc[
        events["experiment_group"].isna(), "user_id"
    ].nunique()
)

invalid_group_rows = int(
    (
        events["experiment_group"].notna()
        & ~events["experiment_group"].isin(valid_groups)
    ).sum()
)

# Do not infer assignment using the mode.
# Build one assignment per user only when every row has a valid,
# consistent group label.
assignment_complete = (
    events.groupby("user_id")["experiment_group"]
    .agg(
        lambda values: (
            values.notna().all()
            and values.isin(valid_groups).all()
            and values.nunique() == 1
        )
    )
)

invalid_assignment_users = int((~assignment_complete).sum())

print(f"Users appearing in multiple groups: {multi_group_users:,}")
print(f"Users with at least one missing assignment event: {missing_group_users:,}")
print(f"Rows with invalid group labels: {invalid_group_rows:,}")
print(f"Users with incomplete or invalid assignments: {invalid_assignment_users:,}")

if multi_group_users or missing_group_users or invalid_group_rows:
    raise ValueError(
        "Assignment validation failed; investigate missing, conflicting, "
        "or invalid group labels before analyzing outcomes."
    )

assignment = (
    events.groupby("user_id")["experiment_group"]
    .first()
)

group_counts = assignment.value_counts()
print(group_counts)

if set(group_counts.index) != valid_groups:
    raise ValueError(
        "Expected exactly control and treatment groups; inspect observed labels."
    )

srm = sample_ratio_mismatch(
    int(group_counts["control"]),
    int(group_counts["treatment"]),
    expected_treatment_share=0.5,
)
srm

A sample-ratio mismatch p-value below the chosen threshold is a warning, not a diagnosis. Investigate eligibility, randomization, logging, and exposure definitions. A non-significant test does not prove randomization was correct.

## 3. Build one row per user and inspect pre-treatment balance proxies
The event table may not identify which attributes were measured before assignment. Treat device and traffic source as covariates only if the product's assignment design confirms they are pre-treatment.

In [ ]:
# Build one row per user using the validated assignment from Cell 6.
user_features = (
    events.groupby("user_id", as_index=False)
    .agg(
        device=(
            "device",
            lambda s: (
                s.dropna().mode().iloc[0]
                if not s.dropna().empty
                else "unknown"
            ),
        ),
        traffic_source=(
            "traffic_source",
            lambda s: (
                s.dropna().mode().iloc[0]
                if not s.dropna().empty
                else "unknown"
            ),
        ),
        purchased=("event_name", lambda s: int((s == "purchase").any())),
    )
)

# Use the validated assignment instead of inferring group labels from event rows.
user_features = user_features.merge(
    assignment.rename("experiment_group"),
    left_on="user_id",
    right_index=True,
    how="inner",
    validate="one_to_one",
)

balance = pd.crosstab(
    user_features["device"],
    user_features["experiment_group"],
    normalize="columns",
)
display(balance)

## 4. Logistic regression adjustment
The model estimates adjusted association between treatment assignment and whether a user had any purchase event. This is a simple teaching model; it does not account for cluster structure, pre-treatment eligibility, or uncertainty from the modeling process. If covariates are post-treatment, remove them.

In [ ]:

model_data = user_features[
    user_features["experiment_group"].isin(["control", "treatment"])
].copy()

model_data["treatment"] = (
    model_data["experiment_group"] == "treatment"
).astype(int)

# Device and traffic source are treated as pre-treatment covariates
# for this synthetic dataset; verify timing before real-world adjustment.
adjusted_results = adjusted_logistic_regression(
    model_data[
        ["purchased", "treatment", "device", "traffic_source"]
    ]
)

print(f"Users included: {adjusted_results['n_users']:,}")
print(
    "Adjusted treatment coefficient (log-odds): "
    f"{adjusted_results['coefficient_log_odds']:.4f}"
)
print(f"Adjusted odds ratio: {adjusted_results['odds_ratio']:.3f}")
print(
    "Odds-ratio 95% confidence interval: "
    f"[{adjusted_results['odds_ratio_ci_low']:.3f}, "
    f"{adjusted_results['odds_ratio_ci_high']:.3f}]"
)
print(f"Treatment coefficient p-value: {adjusted_results['p_value']:.6g}")

print(
    "\nThe odds ratio describes an adjusted association with purchase odds, "
    "not a percentage-point conversion lift. Results are illustrative "
    "because the dataset is synthetic."
)

display(pd.DataFrame([adjusted_results]))


## 5. Multiple testing and final reporting
Use Holm adjustment only for a pre-specified family of tests. The example below illustrates the calculation, not p-values from this dataset.


In [ ]:
example_p_values = [0.01, 0.04, 0.03]
pd.DataFrame({'raw_p_value': example_p_values, 'holm_adjusted_p_value': holm_bonferroni(example_p_values)})

## Reporting checklist
1. State the unit of randomization and primary outcome.
2. Report assigned users, exposures, missing outcomes, and group consistency.
3. Report sample-ratio diagnostics and investigate any imbalance before effect estimation.
4. Report conversion rates, absolute lift, confidence interval, p-value, planned sample size, and stopping rule.
5. Report guardrails and pre-specified multiplicity handling.
6. Explain why covariates are pre-treatment and disclose model limitations.
7. Label all simulated-data conclusions as illustrative, not real product evidence.